In [26]:
pip install easyocr

  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 9.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 10.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 8.2 MB/s eta 0:00:0000:0100:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 9.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 9.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 9.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 9.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 9.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 9.7 MB/s eta 0:00:00:00:0100:02
   ━━━━

In [24]:
import cv2
import numpy as np
import os
import re
import json
from collections import defaultdict
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, PatternFill

# Для работы с PDF
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
Image.MAX_IMAGE_PIXELS = None   # отключаем предупреждение о "бомбе"

pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'




# ------------------- НАСТРОЙКИ (при необходимости поменяйте пути к Tesseract и Poppler) -------------------
# Для Linux/macOS обычно достаточно 'tesseract'
# Для pdf2image может потребоваться указать путь к poppler:
# from pdf2image import convert_from_path; convert_from_path(..., poppler_path=r'C:\path\to\poppler\bin')
# --------------------------------------------------------------------------------------------------------

# ------------------- ПАРСИНГ ПРОЕКТОВ (как раньше) -------------------
SKIPPED_RULE_PATTERNS = ['11MBV', '11MBX', '11MKD']

def is_skipped_rule(project_data):
    if project_data.get('project', {}).get('type') != 'rule':
        return False
    code = project_data.get('project', {}).get('code', '')
    return any(pattern in code for pattern in SKIPPED_RULE_PATTERNS)

def load_projects(folder_path):
    projects = {}
    for filename in os.listdir(folder_path):
        if not filename.endswith('.json'):
            continue
        file_path = os.path.join(folder_path, filename)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print(f"Ошибка чтения {filename}: {e}")
            continue

        if 'project' not in data or 'code' not in data['project']:
            continue
        ptype = data['project'].get('type')
        if ptype not in ('rule', 'parameter'):
            continue
        code = data['project']['code']
        projects[code] = data
    return projects

def get_input_signals(project_data):
    signals = []
    for elem in project_data.get('elements', {}).values():
        if elem.get('type') == 'input-signal':
            name = elem.get('props', {}).get('name')
            if name:
                signals.append(name)
    return signals

def collect_base_signals(project_code, projects, visited=None):
    if visited is None:
        visited = set()
    if project_code in visited:
        return set()
    visited.add(project_code)

    project = projects.get(project_code)
    if not project:
        return set()

    if is_skipped_rule(project):
        return set()

    base_signals = set()
    for signal_name in get_input_signals(project):
        if signal_name in projects:
            base_signals.update(collect_base_signals(signal_name, projects, visited.copy()))
        else:
            base_signals.add(signal_name)
    return base_signals

def build_rule_table(folder_path):
    projects = load_projects(folder_path)
    rule_base_signals = {}
    for code, data in projects.items():
        if data['project']['type'] == 'rule' and not is_skipped_rule(data):
            base_set = collect_base_signals(code, projects)
            rule_base_signals[code] = base_set

    signal_to_rules = defaultdict(list)
    for rule_code, base_set in rule_base_signals.items():
        for signal in base_set:
            signal_to_rules[signal].append(rule_code)

    return {signal: sorted(rules) for signal, rules in sorted(signal_to_rules.items())}


def preprocess_image_for_ocr(image):
    """
    Многоэтапная предобработка изображения для повышения качества OCR.
    Принимает numpy array (RGB или grayscale).
    """
    # Переводим в оттенки серого
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image

    # 1. Удаление шума (медианный фильтр)
    denoised = cv2.medianBlur(gray, 3)

    # 2. Адаптивная пороговая бинаризация (хорошо для неравномерного освещения)
    binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 2)

    # 3. Морфологическое закрытие (убирает мелкие дыры внутри букв)
    kernel = np.ones((1, 1), np.uint8)
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    # 4. Увеличение контраста (если нужно) – опционально
    # enhanced = cv2.equalizeHist(closed)  # может пересветить
    # Лучше: повышение резкости
    kernel_sharpen = np.array([[-1, -1, -1],
                               [-1,  9, -1],
                               [-1, -1, -1]])
    sharpened = cv2.filter2D(closed, -1, kernel_sharpen)

    return sharpened

# ------------------- РАЗБИЕНИЕ KKS НА ЧАСТИ -------------------
def split_kks(kks_code):
    """
    Разбивает KKS-код на осмысленные блоки, например:
    '7L0LBA45CT001' -> ['7', 'L0LBA45', 'CT001']
    '7KKS001' -> ['7', 'KKS001'] или что-то подобное.
    Используем регулярное выражение: цифры в начале, затем буквенно-цифровые блоки по переходу цифра-буква.
    Более простой подход: разбиваем по границам "цифра-буква" и "буква-цифра", но сохраняем последовательность.
    """
    # Удаляем лишние пробелы и переносы
    code = re.sub(r'\s+', '', kks_code)
    # Разбиваем: сначала одна или несколько цифр в начале, затем чередование букв и цифр
    parts = re.findall(r'^\d+|[A-Za-z]+\d+|\d+[A-Za-z]+|[A-Za-z]+', code)
    # Если разбивка не сработала, возвращаем весь код как один элемент
    if not parts:
        return [code]
    return parts

# ------------------- ИЗВЛЕЧЕНИЕ ТЕКСТА ИЗ PDF (С OCR ПРИ НЕОБХОДИМОСТИ) -------------------
def extract_text_from_pdf(pdf_path, dpi=400, use_ocr=True):
    """Извлекает текст из PDF с возможностью OCR для сканированных страниц."""
    # Сначала пробуем стандартное текстовое извлечение
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ''
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + '\n'
        if len(text.strip()) > 100:   # достаточно текста – не запускаем тяжёлый OCR
            return text
    except Exception as e:
        print(f"pdfplumber ошибка {pdf_path}: {e}")

    if not use_ocr:
        return ""

    print(f"Запущен улучшенный OCR для {os.path.basename(pdf_path)}...")
    try:
        # Конвертируем PDF в изображения с высоким DPI
        images = convert_from_path(pdf_path, dpi=dpi, fmt='jpeg')
        full_text = []

        for idx, img in enumerate(images):
            # Конвертируем PIL Image в numpy array для OpenCV
            img_np = np.array(img)
            # Предобработка
            processed_img = preprocess_image_for_ocr(img_np)
            cv2.imwrite(f"debug_page_{idx}_1.png", processed_img)

            # Распознавание с оптимальными параметрами Tesseract
            # --psm 6: блок текста, --oem 1: LSTM engine, -c preserve_interword_spaces=1
            custom_config = r'--oem 1 --psm 12 -c preserve_interword_spaces=1'
            # Языки: русский и английский
            page_text = pytesseract.image_to_string(processed_img, lang='rus+eng+deu', config=custom_config)
            full_text.append(page_text)

        return "\n".join(full_text)
    except Exception as e:
        print(f"Улучшенный OCR не удался для {pdf_path}: {e}")
        return ""

# ------------------- ПОИСК KKS В PDF С УЧЁТОМ РАЗБИЕНИЯ -------------------
def kks_present_in_text(kks_parts, text):
    """
    Проверяет, встречаются ли все части KKS кода в тексте.
    Учитывает, что между частями могут быть пробелы, переносы строк, другие символы.
    """
    # Приводим текст к нижнему регистру и удаляем все пробельные символы (для гибкого поиска)
    # Но для точности лучше искать каждую часть как отдельное слово с возможными пробелами.
    # Проще: проверяем, что каждая часть присутствует как подстрока (без учёта пробелов внутри части).
    text_lower = text.lower()
    # Для каждой части делаем поиск с игнорированием регистра (уже привели текст к нижнему)
    for part in kks_parts:
        part_lower = part.lower()
        if part_lower not in text_lower:
            return False
    return True

def search_kks_in_pdfs(kks_list, pdf_folder):
    """
    Для каждого KKS кода (из списка) ищет его в PDF-файлах.
    Возвращает словарь {kks: [список имён pdf-файлов]}.
    """
    if not os.path.isdir(pdf_folder):
        print(f"Папка с PDF '{pdf_folder}' не найдена.")
        return {kks: [] for kks in kks_list}

    pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print("Нет PDF-файлов.")
        return {kks: [] for kks in kks_list}

    # Предварительно разбиваем все KKS на части
    kks_parts = {kks: split_kks(kks) for kks in kks_list}
    #print(kks_parts)

    # Для каждого PDF извлекаем текст (один раз) и кэшируем
    pdf_text_cache = {}
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_folder, pdf_file)
        print(f"Обработка {pdf_file}...")
        pdf_text_cache[pdf_file] = extract_text_from_pdf(pdf_path)
        print(pdf_text_cache[pdf_file])
        break

    # Поиск
    kks_to_pdfs = defaultdict(list)
    for kks in kks_list:
        parts = kks_parts[kks]
        for pdf_file, text in pdf_text_cache.items():
            if text and kks_present_in_text(parts, text):
                kks_to_pdfs[kks].append(pdf_file)
    return dict(kks_to_pdfs)

# ------------------- СОХРАНЕНИЕ В EXCEL С ОБЪЕДИНЕНИЕМ И ПОДСВЕТКОЙ -------------------
def save_expanded_table(signal_to_rules_dict, signal_to_pdfs, output_file="Миграция.xlsx"):
    rows = []
    for idx, (signal, rules) in enumerate(signal_to_rules_dict.items(), start=1):
        pdfs = signal_to_pdfs.get(signal, [])
        pdfs_str = ', '.join(pdfs) if pdfs else ''
        for rule in rules:
            rows.append({
                '№': idx,
                'Базовый сигнал': signal,
                'Правило': rule,
                'P&ID диаграмма(ы)': pdfs_str
            })

    if not rows:
        print("Нет данных для экспорта.")
        return

    df = pd.DataFrame(rows)
    temp_file = "temp_merged.xlsx"
    df.to_excel(temp_file, index=False, engine='openpyxl')

    wb = load_workbook(temp_file)
    ws = wb.active
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")

    current_signal = None
    start_row = None
    for row_idx in range(2, ws.max_row + 2):
        cell_signal = ws.cell(row=row_idx, column=2).value
        if cell_signal != current_signal:
            if current_signal is not None and start_row is not None:
                end_row = row_idx - 1
                if start_row < end_row:
                    # Объединяем столбцы 1, 2, 4
                    ws.merge_cells(start_row=start_row, start_column=1, end_row=end_row, end_column=1)
                    ws.merge_cells(start_row=start_row, start_column=2, end_row=end_row, end_column=2)
                    ws.merge_cells(start_row=start_row, start_column=4, end_row=end_row, end_column=4)
                    for col in (1, 2, 4):
                        merged_cell = ws.cell(row=start_row, column=col)
                        merged_cell.alignment = Alignment(horizontal='center', vertical='center')
                    if current_signal and "xm" in current_signal.lower():
                        for col in (1, 2, 4):
                            ws.cell(row=start_row, column=col).fill = yellow_fill
            current_signal = cell_signal
            start_row = row_idx

    # Последняя группа
    if current_signal is not None and start_row is not None:
        end_row = ws.max_row
        if start_row < end_row:
            ws.merge_cells(start_row=start_row, start_column=1, end_row=end_row, end_column=1)
            ws.merge_cells(start_row=start_row, start_column=2, end_row=end_row, end_column=2)
            ws.merge_cells(start_row=start_row, start_column=4, end_row=end_row, end_column=4)
            for col in (1, 2, 4):
                merged_cell = ws.cell(row=start_row, column=col)
                merged_cell.alignment = Alignment(horizontal='center', vertical='center')
            if current_signal and "xm" in current_signal.lower():
                for col in (1, 2, 4):
                    ws.cell(row=start_row, column=col).fill = yellow_fill
        else:
            for col in (1, 2, 4):
                cell = ws.cell(row=start_row, column=col)
                cell.alignment = Alignment(horizontal='center', vertical='center')
                if current_signal and "xm" in current_signal.lower():
                    cell.fill = yellow_fill

    wb.save(output_file)
    os.remove(temp_file)
    print(f"Таблица сохранена в {output_file}")    

In [25]:

projects_folder = 'projects'          # папка с JSON проектами
pdf_folder = 'Diagrams'           # папка с P&ID PDF
output_excel = 'Миграция.xlsx'

    # Шаг 1: строим зависимость "базовый сигнал -> правила"
signal_to_rules = build_rule_table(projects_folder)
if not signal_to_rules:
        print("Нет базовых сигналов. Завершение.")
        exit()

    # Шаг 2: поиск базовых сигналов в PDF
print(f"Поиск {len(signal_to_rules)} сигналов в PDF...")
signal_to_pdfs = search_kks_in_pdfs([kks.split('§')[0] for kks in list(signal_to_rules.keys())], pdf_folder)

    # Шаг 3: сохранение результата
#save_expanded_table(signal_to_rules, signal_to_pdfs, output_excel)

Поиск 207 сигналов в PDF...
Обработка ru1003-10mag-mfb030-200444_f.pdf...
Запущен улучшенный OCR для ru1003-10mag-mfb030-200444_f.pdf...
О ООО

р ет р ртр рр

ПВН 5678

renzende Systeme

но)

OINING SYSTEMS

LBS

ND-Anzapfleitun

MAL

Entwässerun

LP-EXTRACTION P

{pe

TURBINE DRA

In

b

>

Bl

Kondensat

inde | leckdampf

ии

LCA

CONDENSATE

MAM

cp

INDLE LEAKOFF STEAM

MAC

ND-Turbine

MAN

Umleitdampf

LP-TURBINE

BYPASS STEAM

Evakuierung

MAW

Wel lendichungsdampf

MAJ

Sperrwasser

GLAND STEAMei tung

Reserve Kondensat

PAB

Kuhlwasserleituna

LCR

STANDBY CONDENSATE

COOL WATER PIPE

Steuer luft

O

o> ©

СЕВ

CONTROL AIR

GM

Betriebsabwasser

(i

CS

PLANT DRAINAGE SYSTEM

[т GMAGIB |

1 MAGIA |

1 MAGIG

or

era

| (203 |

| CPAB2 |

®

GHC

Deionatverteilsystem

EEE

  О

O

DEMINERALIZED WATER

| ИМАМ6е BROBI

DISTRIBUTION SYSTEM

.

12928-986 132

                      .

ehörige Systemschaltpläne

eee

ATED SYSTEM DIAGRAMS

<

GHC2e BROIZ

12928-986130

N ——— ZZ

m — —